In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import joblib

In [22]:
product = pd.read_csv("/Users/aashishmewada/Desktop/PRISM AI/PRISM_AI/data/raw/product_inventory.csv")

In [23]:
product.head()

,ProductID,ProductName,Category,Brand,Price,Rating,Description,Stock
0,PRD-3394,iPhone 15,Electronics,Apple,799.99,4.6,Latest flagship smartphone featuring an advanc...,0
1,PRD-1233,MacBook Air M2,Electronics,Apple,1099.00,4.7,Lightweight and powerful laptop with Apple M2 ...,25
2,PRD-0363,AirPods Pro,Electronics,Apple,249.00,4.2,"Active Noise Cancellation, Adaptive Audio, and...",5
3,PRD-1620,Noise Canceling Headphones,Electronics,Sony,348.00,4.6,Industry-leading noise canceling wireless head...,350
4,PRD-8323,4K OLED TV 55-inch,Electronics,Sony,1398.00,4.1,Stunning 4K OLED picture quality with immersiv...,120


In [24]:
product.shape

(23, 8)

In [25]:
product.isnull().sum()

ProductID      0
ProductName    0
Category       0
Brand          0
Price          0
Rating         0
Description    0
Stock          0
dtype: int64

In [26]:
product["Product_Features"] = (
    product["ProductName"]
    + " "
    +
   product["Category"]
    + " "
    +
   product["Brand"]
    + " "
    +
   product["Description"]
)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [32]:
tfidf = TfidfVectorizer(
    stop_words="english"
)


tfidf_matrix = tfidf.fit_transform(
    product["Product_Features"]
)


tfidf_matrix.shape

(23, 254)

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
pip install -U scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 2.2 MB/s  0:00:03 eta 0:00:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.7.2
    Uninstalling scikit-learn-1.7.2:
      Successfully uninstalled scikit-learn-1.7.2
Note: you may need to restart the kernel to use updated packages.


In [41]:
similarity_matrix = cosine_similarity(
    tfidf_matrix
)

In [46]:
def recommend_products(product_name, n=5):

    index = product[
        product["ProductName"] == product_name
    ].index[0]


    similarity_scores = list(
        enumerate(
            similarity_matrix[index]
        )
    )


    sorted_products = sorted(
        similarity_scores,
        key=lambda x:x[1],
        reverse=True
    )


    recommendations=[]


    for i,score in sorted_products[1:n+1]:

        recommendations.append({

            "Product":
            product.iloc[i]["ProductName"],

            "Brand":
            product.iloc[i]["Brand"],

            "Category":
            product.iloc[i]["Category"],

            "Similarity Score":
            round(score,3)

        })


    return pd.DataFrame(
        recommendations
    )

In [47]:
recommend_products(
    "AirPods Pro",
    5
)

,Product,Brand,Category,Similarity Score
0,Noise Canceling Headphones,Sony,Electronics,0.108
1,MacBook Air M2,Apple,Electronics,0.091
2,iPhone 15,Apple,Electronics,0.066
3,Galaxy S24,Samsung,Electronics,0.023
4,Mechanical Wireless Keyboard,Logitech,Electronics,0.020


In [49]:
import os

In [50]:
os.makedirs(
    "models",
    exist_ok=True
)

In [52]:
joblib.dump(
    product,
    "models/products.pkl"
)


joblib.dump(
    tfidf,
    "models/tfidf_vectorizer.pkl"
)


joblib.dump(
    similarity_matrix,
    "models/recommendation_similarity.pkl"
)

['models/recommendation_similarity.pkl']